In [1]:
!python train.py \
-data_dir dataset/training_dataset/train/ \
-pretrained pretrained_weights/pretrained_weights_from_clear_image.pt \
-ENCODER resnet101 \
-n_epoch 10 \
-batch_size 32 \
-augment \
-rect_training
#During training, the directory `dataset/training_dataset/train/masks/train` is continuously updated and written to.

/home/xuechunliang/anaconda3/envs/xuechunliang/lib/python3.9/site-packages/requests/__init__.py:109: RequestsDependencyWarning: urllib3 (2.13.900) or chardet (5.2.0)/charset_normalizer (3.3.2) doesn't match a supported version!
  warnings.warn(
/home/xuechunliang/anaconda3/envs/xuechunliang/lib/python3.9/site-packages/albumentations/__init__.py:24: UserWarning: A new version of Albumentations is available: 2.0.8 (you have 1.4.20). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()
Using CUDA device0 _CudaDeviceProperties(name='NVIDIA RTX 6000 Ada Generation', total_memory=48640MB)

total images = 1256
Loading pretrained checkpoint: pretrained_weights/pretrained_weights_from_clear_image.pt
0.005
Epoch 0:  15%|▍  | 96/640 [00:21<01:51,  4.88it/s, train_loss=0.06433, size=448]

In [ ]:
#Restore the updated pseudo-labels.During dataset creation, each original degraded image covers an area of approximately 4 × 4 meters. 
#For model training, each image is cropped into four 2 × 2 meter patches. Therefore, after updating the pseudo-labels, 
#the four cropped patches need to be stitched back together to restore the original 4 × 4 meter size.

import os
from PIL import Image

# ==============================
# Input / Output directories
# ==============================
degraded_image_dir = "dataset/preprocessing/degraded_image/"   # original degraded images
patch_mask_dir     = "dataset/training_dataset/train/masks/train/"                         # predicted mask patches
output_dir         = os.path.join(patch_mask_dir, "updated_degraded_image_label")  # merged full-size labels

os.makedirs(output_dir, exist_ok=True)

# Sorted file lists
mask_patch_names   = sorted([f for f in os.listdir(patch_mask_dir) if f.lower().endswith(".jpg")])
original_img_names = sorted([f for f in os.listdir(degraded_image_dir) if f.lower().endswith(".jpg")])

# ==============================
# Merge every 4 patches → 1 full label image (2×2 grid)
# ==============================
img_index = 0

for i in range(0, len(mask_patch_names), 4):

    patch_group = mask_patch_names[i:i+4]
    if len(patch_group) < 4:
        break  # skip incomplete groups

    # Load and resize patches to 480×480
    patch_images = [
        Image.open(os.path.join(patch_mask_dir, name)).resize((480, 480))
        for name in patch_group
    ]

    # Create a 960×960 canvas (white background)
    merged_image = Image.new("RGB", (960, 960), color=(255, 255, 255))

    # Paste patches in row-major order
    merged_image.paste(patch_images[0], (0, 0))       # top-left
    merged_image.paste(patch_images[1], (480, 0))     # top-right
    merged_image.paste(patch_images[2], (0, 480))     # bottom-left
    merged_image.paste(patch_images[3], (480, 480))   # bottom-right

    # Use the corresponding original degraded image filename
    output_name = original_img_names[img_index]
    img_index += 1

    save_path = os.path.join(output_dir, output_name)
    merged_image.save(save_path)

print("Merging finished. Output filenames match the original degraded images.")
